In [0]:
%pip install Faker

dbutils.library.restartPython()

In [0]:
import sys
from pathlib import Path

current = Path.cwd()
repo_root = None

for path in [current, *current.parents]:
    if (
        (path / "src" / "data_generator").exists()
        and
        (path / "databricks" / "ingestion" / "state_updater.py").exists()
    ):
        repo_root = path
        break

if repo_root is None:
    raise RuntimeError("Could not find AI-Analytics-Data-Platform repo root")

for import_path in [
    repo_root,
    repo_root / "databricks",
]:
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

print("Repo root:", repo_root)

In [0]:
import importlib
from datetime import date

import ingestion.delta_state as delta_state
import ingestion.state_updater as state_updater
import src.data_generator.daily_generation as daily_generation

importlib.reload(delta_state)
importlib.reload(state_updater)
importlib.reload(daily_generation)

SIMULATION_DATE = date(2026, 8, 23)
SEED = 42

TEST_SCHEMA = "ai_analytics.state_updater_test"

print("Modules loaded")

In [0]:
state = delta_state.load_generator_state(spark)

clean_records = daily_generation.generate_daily_records(
    state,
    SIMULATION_DATE,
    SEED,
)

validation = daily_generation.validate_daily_records(
    clean_records,
    state,
    SIMULATION_DATE,
)

print(validation)

for entity in [
    "user_updates",
    "conversations",
    "messages",
    "errors",
    "subscriptions",
]:
    print(f"{entity}: {len(clean_records[entity]):,}")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TEST_SCHEMA}")

print("Created test schema:", TEST_SCHEMA)

In [0]:
TABLES_TO_CLONE = [
    "generator_users",
    "generator_user_state",
    "generator_recent_conversations",
    "generator_recent_messages",
    "generator_subscriptions",
    "generator_daily_counts",
]

for table in TABLES_TO_CLONE:
    target = f"{TEST_SCHEMA}.{table}"
    source = f"ai_analytics.source.{table}"

    spark.sql(f"DROP TABLE IF EXISTS {target}")

    spark.sql(f"""
        CREATE TABLE {target}
        SHALLOW CLONE {source}
    """)

    print(
        f"{table}:",
        spark.table(target).count(),
        "rows"
    )

In [0]:
from pyspark.sql import functions as F

count_table = spark.table(
    f"{TEST_SCHEMA}.generator_daily_counts"
)

display(
    count_table
    .groupBy("entity")
    .agg(
        F.min("activity_date").alias("first_date"),
        F.max("activity_date").alias("last_date"),
        F.countDistinct("activity_date").alias("days"),
    )
    .orderBy("entity")
)

In [0]:
validation_summary = state_updater.validate_state_update(
    spark,
    clean_records,
    SIMULATION_DATE,
    source_schema=TEST_SCHEMA,
)

validation_summary

In [0]:
assert validation_summary["previous_generator_date"] == date(2026, 8, 22)
assert validation_summary["mode"] == "new_day"
assert validation_summary["window_start"] == date(2026, 7, 25)
assert validation_summary["mutation_attempted"] is False

print("PASS: Aug 23 state update validates without mutation")

In [0]:
update_summary = state_updater.update_generator_state(
    spark,
    clean_records,
    SIMULATION_DATE,
    source_schema=TEST_SCHEMA,
)

update_summary

In [0]:
assert update_summary["completed"] is True
assert update_summary["mutation_attempted"] is True
assert update_summary["mode"] == "new_day"

print("PASS: cloned generator state advanced to Aug 23")

In [0]:
before_retry = {}

checks = [
    ("generator_user_state", "user_id"),
    ("generator_recent_conversations", "conversation_id"),
    ("generator_recent_messages", "message_id"),
    ("generator_subscriptions", "subscription_id"),
    ("generator_daily_counts", None),
]

for table, key in checks:
    df = spark.table(f"{TEST_SCHEMA}.{table}")

    before_retry[table] = df.count()

    print(
        table,
        "rows=",
        before_retry[table],
    )

In [0]:
retry_summary = state_updater.update_generator_state(
    spark,
    clean_records,
    SIMULATION_DATE,
    source_schema=TEST_SCHEMA,
)

retry_summary

In [0]:
assert retry_summary["mode"] == "retry"
assert retry_summary["completed"] is True

for table, key in checks:
    df = spark.table(f"{TEST_SCHEMA}.{table}")

    after_count = df.count()

    print(
        f"{table}: "
        f"before={before_retry[table]:,}, "
        f"after={after_count:,}"
    )

    assert after_count == before_retry[table]

    if key is not None:
        assert df.select(key).distinct().count() == after_count

print("\nPASS: same-day retry is idempotent")

In [0]:
aug23_count_rows = (
    spark.table(f"{TEST_SCHEMA}.generator_daily_counts")
    .filter(
        F.col("activity_date")
        == F.lit("2026-08-23").cast("date")
    )
)

display(aug23_count_rows.orderBy("entity"))

assert aug23_count_rows.count() == 4

print("PASS: Aug 23 daily-count rows were not duplicated")

In [0]:
spark.sql(f"DROP SCHEMA IF EXISTS {TEST_SCHEMA} CASCADE")

print("Temporary state updater test schema removed")